# Notbook 005 — ¿Es necesario GELU para aprender lenguaje?

**Objetivo:** evaluar operaciones neuronales por su capacidad de predecir texto español; NO por XOR, círculos ni lunas.

Filosofía: cuestionar → eliminar → simplificar → acelerar → automatizar.

**Fuentes:** el tokenizador usa la función `silabas` original de [SiriusMagic/Notbook](https://github.com/SiriusMagic/Notbook/blob/main/laboratorio/silabas.py). El conjunto es `SetFit/amazon_reviews_multi_es`, ya empleado en el banco original. No se modifica ninguno de los dos repositorios de origen.

**Arquitecturas controladas:** Transformer causal de 2 bloques, embedding compartido con la proyección de salida y atención idéntica. Solo cambia la FFN:
1. GELU convencional.
2. ReLU² (`relu(x)**2`), mismo número de parámetros.
3. Sin FFN: prueba de eliminación radical, con menos parámetros.

**Criterio:** menor BPC en texto normalizado con particiones de entrenamiento, validación y prueba separadas. El BPC incluye el token de fin de documento. No mide automáticamente razonamiento ni comprensión; requiere evaluaciones posteriores.

**Importante:** el resultado previo de clasificación de reseñas NO demuestra superioridad del tokenizador en modelado causal de lenguaje. Usamos las mismas sílabas para aislar el efecto de la neurona. No se reclama originalidad de ReLU², GELU ni de la arquitectura Transformer.

**Limitaciones:** conjunto pequeño, contexto limitado, solo texto normalizado, prueba sintética inicial no concluyente, velocidad y resultados sensibles al hardware. La validez estadística requerirá más semillas, más datos y comparación a igual cómputo.

In [ ]:
# Preparación: clonar los dos repositorios públicos y instalar la dependencia de datos.
import os, sys, subprocess
from pathlib import Path
base = Path("/content") if Path("/content").exists() else Path.cwd()
for repo in ("Notbook", "Neuron"):
    dest = base / repo
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/SiriusMagic/{repo}.git", str(dest)], check=True)
try:
    import datasets
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
os.chdir(base / "Neuron")
env = os.environ.copy()
env["PYTHONPATH"] = str(base / "Notbook") + os.pathsep + env.get("PYTHONPATH", "")
print("Fuentes:", base / "Notbook" / "laboratorio" / "silabas.py")
print("Código:", base / "Neuron" / "experiments" / "language005.py")

## 1. Prueba de funcionamiento (no es evaluación científica)

Confirma que el código entrena y guarda CSV. Usa frases sintéticas, por lo que sus cifras **no demuestran ninguna mejora**.

In [ ]:
subprocess.run(
    [sys.executable, "experiments/language005.py", "--smoke", "--steps", "3", "--seeds", "11"],
    check=True, env=env
)

## 2. Evaluación principal con reseñas reales en español

Se descargan textos públicos de entrenamiento y prueba de la misma fuente que utiliza Notbook. Validación se separa de entrenamiento; la prueba oficial se evalúa al final después de elegir el mejor paso por validación. No hay sustitución silenciosa por texto sintético.

El vocabulario se construye **solo con entrenamiento**. Las sílabas desconocidas se codifican mediante caracteres; el sistema comprueba que el texto normalizado se pueda reconstruir.

Tres semillas para exploración. Para confirmar una ventaja se necesitan al menos cinco y presupuestos de entrenamiento adicionales.

In [ ]:
# Ejecutar cuando se desee realizar la prueba real.
RUN_FULL = True
if RUN_FULL:
    subprocess.run(
        [sys.executable, "experiments/language005.py",
         "--steps", "180", "--seeds", "11", "23", "47",
         "--train-docs", "2500", "--val-docs", "350", "--test-docs", "350"],
        check=True, env=env
    )

## 3. Lectura de resultados

- **BPC_prueba:** menor es mejor (texto español normalizado). Para comparar aquí, todos usan el mismo tokenizador.
- **Parámetros:** GELU y ReLU² tienen el mismo número; `eliminada` tiene menos.
- **BPC_validacion:** se utiliza para elegir el mejor punto de entrenamiento.
- **segundos_entrenar:** informativo, dependiente de hardware, no representa un benchmark de inferencia.
- **No declarar éxito** con una sola semilla ni con diferencias pequeñas. Revisar dispersión, convergencia y longitud de entrenamiento.

La prueba definitiva debe ampliar corpus, semillas, presupuesto FLOPs y medir latencia de inferencia en el dispositivo objetivo.

In [ ]:
from pathlib import Path
import pandas as pd
ruta = Path("results/005/metricas.csv")
if ruta.exists():
    tabla = pd.read_csv(ruta)
    display(tabla)
    display(tabla.groupby("modelo")[["bpc_prueba", "parametros", "segundos_entrenar"]].agg(["mean", "std"]))
else:
    print("Todavía no se ha ejecutado la evaluación real.")

## 4. Conclusión a completar después de ejecutar

No se añade una operación por obtener un resultado favorable en una tarea pequeña. Se conserva únicamente si mejora la predicción de texto de prueba o los recursos necesarios bajo controles razonables. El código fuente está en `experiments/language005.py`.